# ML-03 — Frame My Lane as an ML Task

This notebook frames a content-refresh prioritization problem using the FlyRank
starter dataset.

The goal is not to predict Google's algorithm. The goal is to support an editor's
decision about **which content items should be reviewed first**.

This notebook focuses on the decision, action, target/proxy, success metric, unit
of analysis, and why an ML-based approach may be useful.

## 1. My lane as an ML task (type)

### Task type: Ranking / Scoring

My ML task is a **ranking/scoring** problem.

The decision is:

> **Which content pages should an editor review or refresh first?**

The output would be a priority score for each content item. Pages with higher
scores would appear earlier in an editor's review queue.

The person acting on the output is a content/SEO editor. The action supported by
the score is to decide which pages deserve attention first.

This is better framed as ranking/scoring rather than simple classification because
the practical decision is not only whether a page is declining. The editor has
limited time, so the useful output is an ordered list of pages to review first.

A wrong prioritization can waste editor time by reviewing a page that does not
need attention, while a missed high-priority page can delay a useful content
refresh.

The starter dataset contains multiple signals such as impressions, clicks,
sessions, engagement, average position, content age, freshness, and recent
30-day activity. These interacting signals make the prioritization problem more
complex than a single fixed if-statement.

For this assignment, the available `is_declining_label` is treated as a
**proxy target** for the priority decision. It is important to note that this
label is rule-derived from `trend_direction`, so it should not be described as
an independently observed future outcome.

In [1]:
# Basic setup for the notebook

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Python environment ready.")

Python environment ready.


## 4. The unit of analysis, as a real dataframe

### Unit of analysis: one content item/page

The starter dataset has **one row per pseudonymized content item**.

Therefore:

> **One row = one content page/item for one pseudonymized client.**

The `content_id` identifies the content item and `client_id` identifies the
pseudonymized client.

The dataset contains content metadata, keyword context, 90-day performance
aggregates, recent 30-day comparison windows, and derived rates/tiers.

For the framing exercise, the important point is that the decision is made at
the **content-item level**: each row represents one page that could potentially
be placed into an editor's review queue.

In [3]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

display(df.head())

Dataset loaded successfully!
Rows: 30,000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
# Check that the expected proxy and leakage-related columns exist.

required_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print("Required framing columns are present.")

if "is_declining_label" in df.columns:
    print("is_declining_label already exists in the starter CSV.")
else:
    # The data dictionary defines this label from trend_direction.
    df["is_declining_label"] = (
        df["trend_direction"].astype(str).str.lower() == "down"
    ).astype(int)
    print("is_declining_label was recreated from trend_direction.")

Required framing columns are present.
is_declining_label was recreated from trend_direction.


In [5]:
# Inspect the proxy target.

target_summary = pd.DataFrame({
    "count": [len(df)],
    "declining_count": [df["is_declining_label"].sum()],
    "non_declining_count": [(df["is_declining_label"] == 0).sum()],
    "declining_rate_pct": [
        df["is_declining_label"].mean() * 100
    ]
})

target_summary

,count,declining_count,non_declining_count,declining_rate_pct
0,30000,16262,13738,54.206667


## 3. Success metric

### Primary metric: Precision@K

The primary success metric will be **Precision@K**.

The practical use case is an editor who can only review a limited number of
pages. Therefore, the most important question is:

> Of the top K pages placed at the beginning of the review queue, how many are
> actually marked as declining by the available proxy?

For example, Precision@50 would measure the proportion of the first 50 ranked
pages that have `is_declining_label = 1`.

A higher Precision@K means that the limited review capacity is concentrated more
heavily on pages that match the defined decline proxy.

For this framing exercise, I would use **K = 50** because it represents a small,
actionable review queue.

This metric is appropriate because the task is ranking/scoring rather than simply
predicting a yes/no label. It directly connects the model output to the editor's
limited review capacity.

However, because the current target is rule-derived, Precision@K should be
interpreted as performance against the proxy label, not as proof that the
selected pages will recover after a content refresh.

In [6]:
# Define the success metric before any modeling.

K = 50

def precision_at_k(y_true, scores, k=50):
    """
    Precision@K for a ranked list.

    y_true:
        Binary target values where 1 means the item matches the proxy target.

    scores:
        Ranking/priority scores. Higher scores are considered higher priority.

    k:
        Number of top-ranked items considered.
    """
    if k <= 0:
        raise ValueError("k must be greater than 0.")

    k = min(k, len(y_true))

    order = np.argsort(-np.asarray(scores))[:k]
    top_k_labels = np.asarray(y_true)[order]

    return top_k_labels.mean()


print(f"Primary success metric: Precision@{K}")
print("Higher is better.")

Primary success metric: Precision@50
Higher is better.


## 4. The unit of analysis, as a real dataframe

### Unit of analysis: one content item/page

The starter dataset has **one row per pseudonymized content item**.

Therefore:

> **One row = one content page/item for one pseudonymized client.**

The `content_id` identifies the content item and `client_id` identifies the
pseudonymized client.

The dataset contains content metadata, keyword context, 90-day performance
aggregates, recent 30-day comparison windows, and derived rates/tiers.

For the framing exercise, the important point is that the decision is made at
the **content-item level**: each row represents one page that could potentially
be placed into an editor's review queue.

In [7]:
# Show the actual dataframe representing the unit of analysis.

display(
    df[
        [
            "content_id",
            "client_id",
            "content_type",
            "content_age_days",
            "days_since_last_update",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "avg_position",
            "trend_direction",
            "is_declining_label",
        ]
    ].head(10)
)

,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,29,17,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,7,9,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,11,11,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,58,78,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,24,145,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,1,5,8.5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,0,1,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,1,28,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,32574,29,68,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,1240,2,3,4.9,down,1


In [8]:
# Verify the basic grain of the dataset.

grain_check = {
    "total_rows": len(df),
    "unique_content_ids": df["content_id"].nunique(),
    "unique_clients": df["client_id"].nunique(),
    "duplicate_content_ids": df["content_id"].duplicated().sum(),
}

pd.Series(grain_check)

,0
total_rows,30000
unique_content_ids,30000
unique_clients,32
duplicate_content_ids,0


### Grain check

The `content_id` should be unique in this starter dataset. This supports the
interpretation that the row-level unit of analysis is a content item/page.

The `client_id` is useful for grouping and analysis, but it should not be treated
as a predictive feature because it is only a pseudonymous identifier.

In [9]:
# Sketch the target column and inspect its relationship to the label source.

target_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

display(df[target_columns].head(10))

print("\nTarget/proxy value counts:")
display(df["is_declining_label"].value_counts(dropna=False).sort_index())

print("\nCross-check against trend_direction:")
display(
    pd.crosstab(
        df["trend_direction"],
        df["is_declining_label"],
        margins=True
    )
)

,trend_direction,trend_pct,is_declining_label
0,down,-41.4,1
1,down,-57.7,1
2,down,-60.9,1
3,stable,-13.8,0
4,down,-34.7,1
5,down,-38.9,1
6,down,-92.3,1
7,stable,0.6,0
8,down,-58.8,1
9,down,-29.2,1



Target/proxy value counts:


,count
is_declining_label,
0,13738
1,16262



Cross-check against trend_direction:


is_declining_label,0,1,All
trend_direction,,,
down,0,16262,16262
flat,1152,0,1152
new,2236,0,2236
stable,5962,0,5962
up,4388,0,4388
All,13738,16262,30000


## 5. Why ML beats a fixed rule here

A simple fixed rule could prioritize pages using one threshold, such as:

> "Review pages where impressions are below a certain value."

However, content performance depends on several interacting signals. A page may
have high impressions but poor click-through performance, an old last-update
date, declining recent activity, weak engagement, or a different content type.

A single threshold would therefore ignore important combinations of signals.

An ML ranking approach can combine multiple available signals and learn patterns
from historical examples rather than relying on one manually chosen cutoff.

The value of ML here is therefore not that it automatically knows which content
Google prefers. Its value is that it can provide a more flexible prioritization
score from multiple measurable signals.

There is still an important limitation: the current starter target is a
rule-derived decline label. Therefore, this notebook only frames the problem
and does not claim that ML has demonstrated a causal or future performance
improvement.

The appropriate claim at this stage is:

> **The proposed ML system is decision-support for prioritizing content review,
> using the available decline label as a teaching proxy.**

In [10]:
# Show several measurable signals that could contribute to prioritization.

candidate_signals = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "engagement_rate",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "impressions_last_30d",
    "impressions_prev_30d",
]

available_signals = [
    col for col in candidate_signals
    if col in df.columns
]

signal_summary = df[available_signals].describe().T

display(signal_summary)

,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
clicks_90d,30000.0,16.097333,75.076958,0.0,0.0,1.00,7.00,4178.0
sessions_90d,30000.0,37.066633,107.069131,1.0,2.0,7.00,27.00,4345.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
engagement_rate,30000.0,2.534520,8.310096,0.0,0.0,0.00,1.35,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0
days_since_last_update,30000.0,46.098300,42.078709,1.0,20.0,20.00,104.00,373.0
impressions_last_30d,30000.0,1429.058733,5643.852081,0.0,10.0,139.00,768.00,238796.0
impressions_prev_30d,30000.0,1783.078500,6150.429511,0.0,19.0,210.00,1143.00,218786.0
